In [1]:
# importing libraries

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import duckdb
from collections import Counter
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

### Import Datasets

In [2]:
# reading datasets and combining them

df_cat = pd.read_csv('data/category_tree.csv')
df_event = pd.read_csv('data/events.csv')
df_item_prop_1 = pd.read_csv('data/item_properties_part1.csv')
df_item_prop_2 = pd.read_csv('data/item_properties_part2.csv')

# combining the both item prop dfs
df_item_prop = pd.concat([df_item_prop_1, df_item_prop_2], ignore_index=True)

del df_item_prop_1, df_item_prop_2

print(df_cat.shape, df_event.shape, df_item_prop.shape)

(1669, 2) (2756101, 5) (20275902, 4)


### Preprocess Datasets

In [3]:
# setting the parent to cat_id itself wherever nulls

# replacing the NaNs in the parentid 
print('before: ')
print(df_cat.isna().sum())
df_cat.loc[df_cat.parentid.isnull(),'parentid'] = df_cat.loc[df_cat.parentid.isnull(), 'categoryid']
print('After: ')
print(df_cat.isna().sum())

print("----------------------------------------------------")

# dropping duplicates from df_event
print('before: ')
print(df_event.duplicated().sum())
df_event.drop_duplicates(inplace=True)
print('After: ')
print(df_event.duplicated().sum())

print("----------------------------------------------------")

# replacing the timestamp to proper form
print('before: ')
print(df_event.timestamp[0:2])
print(df_item_prop.timestamp[0:2])

df_event['timestamp'] = pd.to_datetime(df_event['timestamp'], unit='ms') #.dt.strftime('%Y-%m-%d %H:%M:%S')
df_item_prop['timestamp'] = pd.to_datetime(df_item_prop['timestamp'], unit='ms') #.dt.strftime('%Y-%m-%d %H:%M:%S')

print('After: ')
print(df_event.timestamp[0:2])
print(df_item_prop.timestamp[0:2])

before: 
categoryid     0
parentid      25
dtype: int64
After: 
categoryid    0
parentid      0
dtype: int64
----------------------------------------------------
before: 
460
After: 
0
----------------------------------------------------
before: 
0    1433221332117
1    1433224214164
Name: timestamp, dtype: int64
0    1435460400000
1    1441508400000
Name: timestamp, dtype: int64
After: 
0   2015-06-02 05:02:12.117
1   2015-06-02 05:50:14.164
Name: timestamp, dtype: datetime64[ms]
0   2015-06-28 03:00:00
1   2015-09-06 03:00:00
Name: timestamp, dtype: datetime64[ms]


### Sessionization of events

In [4]:
sessionization_query = """
WITH ordered AS (
    select 
    *, 
    LAG(timestamp) OVER (
            PARTITION BY visitorid
            ORDER BY sequence_no
        ) AS prev_ts,
    from (
    SELECT
        *,
        row_number() OVER (
            PARTITION BY visitorid
            ORDER BY timestamp, itemid, coalesce(transactionid, -1)
        ) AS sequence_no
    FROM df_event
    ) t1
),

flagged AS (
    SELECT
        *,
        CASE
            WHEN prev_ts IS NULL THEN 1
            WHEN timestamp - prev_ts >= INTERVAL '30 minutes' THEN 1
            ELSE 0
        END AS is_new_session
    FROM ordered
),

sessionized AS (
    SELECT
        *,
        SUM(is_new_session) OVER (
            PARTITION BY visitorid
            ORDER BY sequence_no
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS session_id
    FROM flagged
)

SELECT
    *,
    MIN(timestamp) OVER (PARTITION BY visitorid, session_id) AS start_st,
    MAX(timestamp) OVER (PARTITION BY visitorid, session_id) AS end_st,
    ROW_NUMBER() OVER (
        PARTITION BY visitorid, session_id
        ORDER BY sequence_no
    ) AS event_pos_in_session
FROM sessionized
ORDER BY visitorid, sequence_no;
"""

In [5]:
# events_df with sessions
sessionized_events_df = duckdb.query(sessionization_query).to_df()

all_cols = list(sessionized_events_df.columns) 
all_cols.remove('prev_ts')

sessionized_events_df = sessionized_events_df[all_cols]


### CG evaluation 

In [10]:
# since the events df is now sorted, lets take out 10000 random samples from the df 

# we will exclude the 1st event of every user's life so that we atleast have 1 data point to get the recommendations 
first_session_first_event = ~((sessionized_events_df.session_id == 1) & (sessionized_events_df.is_new_session == 1))

# this consists 10000 events with atleast 1 previous interaction of user
query_base = sessionized_events_df.loc[first_session_first_event,:].sample(n=10000)

In [11]:
def build_query_table(query_df, session_df, k=10):
    output_rows = []

    # group once for speed
    user_groups = {
        user_id: grp.sort_values("sequence_no").reset_index(drop=True)
        for user_id, grp in session_df.groupby("visitorid")
    }

    for query_id, (_, row) in enumerate(query_df.iterrows()):
        user_df = user_groups[row["visitorid"]]

        curr_seq = row["sequence_no"]
        curr_sess = row["session_id"]
        curr_pos = row["event_pos_in_session"]
        curr_ts = row["timestamp"]

        # past history across all user history
        hist_df = user_df[user_df["timestamp"] < curr_ts]

        last_k_items = hist_df.tail(k)["itemid"].tolist()

        past_events = hist_df.groupby("event").size()

        # future positives within same session, using only future/current views
        fut_df = user_df[
            (user_df["session_id"] == curr_sess) &
            (user_df["timestamp"] >= curr_ts) &
            (user_df["event"] == "view")
        ]

        future_positives = list(dict.fromkeys(fut_df["itemid"].tolist()))

        if len(future_positives) == 0:
            continue

        output_rows.append({
            "query_id": query_id,
            "visitorid": row["visitorid"],
            "session_id": row["session_id"],
            "sequence_no": curr_seq,
            "event_pos_in_session": curr_pos,
            "query_time": row["timestamp"],
            "current_event": row["event"],
            "last_k_items": last_k_items,
            "past_view_count": past_events.get("view", 0),
            "past_atc_count": past_events.get("addtocart", 0),
            "past_order_count": past_events.get("transaction", 0),
            "future_positives": future_positives,
        })

    return pd.DataFrame(output_rows)

In [12]:
# remove duplicates basis ('visitorid','session_id','timestamp') so that we keep only one entry per combination for testing
query_base.drop_duplicates(subset=['visitorid','session_id','timestamp'],inplace=True)

query_input = build_query_table(query_base, sessionized_events_df)

In [ ]:
#### Evaluation on basic CGs

In [13]:
query_input

,query_id,visitorid,session_id,sequence_no,event_pos_in_session,query_time,current_event,last_k_items,past_view_count,past_atc_count,past_order_count,future_positives
0,0,974226,6.0,226,65,2015-07-15 16:44:33.459,view,"[409334, 409334, 462587, 463002, 463002, 25561...",194,24,7,"[153909, 207602, 338427, 246247, 257308, 22268..."
1,1,563488,1.0,7,7,2015-06-30 02:24:18.385,view,"[132505, 132505, 132505, 132505, 132505, 132505]",6,0,0,[132505]
2,2,776865,2.0,3,1,2015-07-17 23:11:57.915,view,"[429571, 291285]",2,0,0,[76388]
3,3,89526,2.0,2,1,2015-09-10 16:19:01.031,view,[275241],1,0,0,"[275241, 87547]"
4,4,163561,5.0,219,64,2015-05-04 16:37:26.136,view,"[200505, 200505, 212840, 23486, 23486, 354881,...",208,7,3,"[399592, 119736, 461915, 84160, 316140, 245168..."
...,...,...,...,...,...,...,...,...,...,...,...,...
9734,9995,201445,1.0,4,4,2015-05-31 01:57:54.013,view,"[325690, 459048, 430890]",3,0,0,[430890]
9735,9996,315760,1.0,2,2,2015-06-01 07:37:43.055,view,[231656],1,0,0,"[235741, 195020]"
9736,9997,1352525,1.0,8,8,2015-06-21 23:28:01.803,view,"[242245, 451723, 198446, 38515, 198446, 451723...",7,0,0,[408504]
9737,9998,983154,5.0,23,2,2015-06-19 10:46:43.005,view,"[81166, 81166, 329311, 329311, 103287, 103287,...",15,7,0,"[109656, 187138]"


### Storing the data sets

In [ ]:
# storing basic updated tables back

transactions_df = (
    sessionized_events_df[sessionized_events_df.transactionid.notnull()]
)

df_cat.to_csv('data/intermediate_files/category_input.csv', index=False)
sessionized_events_df.to_csv('data/intermediate_files/sessionized_events.csv', index=False)
df_item_prop.to_csv('data/intermediate_files/item_properties.csv', index=False)
transactions_df.to_csv('data/intermediate_files/orders_sorted.csv', index=False)
query_input.to_csv('data/intermediate_files/cg_input_query_10k.csv', index=False)

### Other Queries

In [ ]:
# older query table - not wrong though, needs slight changes and inefficent

# def build_query_table(df, session_df):

#     query_id = 0
#     output_df = []
    
#     for _, row in df.iterrows():
#         new_row = []

#         new_row.append(query_id)                               #query_id
#         new_row.append(row['visitorid'])                       #visitorid
#         new_row.append(row['session_id'])                      #session_id
#         new_row.append(row['timestamp'])                       #query_time

#         user_df = session_df.loc[(session_df.visitorid == row['visitorid'])]

#         # k = 10
#         pre_item_list = user_df.loc[(user_df.timestamp < row['timestamp'])].nlargest(10, 'timestamp')['itemid'].to_list()
#         new_row.append(pre_item_list)                          #last k items - can be empty as well

#         past_events = user_df.loc[(user_df.timestamp < row['timestamp'])].groupby('event').size()
        
#         new_row.append(past_events.get('view', 0))             #past view count
#         new_row.append(past_events.get('addtocart', 0))        #past atc count
#         new_row.append(past_events.get('transaction', 0))      #past order count

        
#         future_positives = user_df.loc[(user_df.timestamp >= row['timestamp']) & (user_df.session_id == row['session_id']),'itemid'].to_list()
#         new_row.append(list(dict.fromkeys(future_positives)))  # future positives

#         output_df.append(new_row)
#         query_id += 1

#     new_table_columns = ['query_id', 'visitorid', 'session_id', 'query_time', 'last_k_items', 'past_view_count', 'past_atc_count', 'past_order_count', 'future_positives']
#     query_table = pd.DataFrame(output_df, columns= new_table_columns)

#     return query_table
